In [12]:
import pandas as pd
import os
import logging
from datetime import datetime


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger('extract')

def extract_data(sensor_file_path, failure_file_path, output_dir='../data/processed/extracted_data'):
    try:
    
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
            logger.info(f"Répertoire créé: {output_dir}")
        

        logger.info(f"Lecture des capteurs : {sensor_file_path}")
        sensor_data = pd.read_csv(sensor_file_path)
        
        logger.info(f"Lecture des pannes : {failure_file_path}")
        failure_data = pd.read_csv(failure_file_path)
        
    
        # On sauvegarde les fichiers extraits pour l'étape de nettoyage
        sensor_data.to_csv(os.path.join(output_dir, 'sensors_extracted.csv'), index=False)
        failure_data.to_csv(os.path.join(output_dir, 'failures_extracted.csv'), index=False)
        
        logger.info(f"Fichiers sauvegardés avec succès dans {output_dir}")
        return sensor_data, failure_data
    
    except Exception as e:
        logger.error(f"Erreur lors de l'extraction : {str(e)}")
        raise

if __name__ == "__main__":
    
    SENSOR_FILE = "../data/raw (1)/predictive_maintenance_sensor_data (1).csv"
    FAILURE_FILE = "../data/raw (1)/predictive_maintenance_failure_logs (1).csv"
    OUTPUT_DIR = "../data/processed/extracted_data"
    

    sensor_df, failure_df = extract_data(SENSOR_FILE, FAILURE_FILE, output_dir=OUTPUT_DIR)
    
    print("\n✅ EXTRACTION RÉUSSIE !")
    print(f"Lignes capteurs chargées : {len(sensor_df)}")
    print(f"Lignes pannes chargées : {len(failure_df)}")
    print("\n--- Aperçu des données ---")
    print(sensor_df.head(3))

2026-02-20 16:22:23,377 - extract - INFO - Répertoire créé: ../data/processed/extracted_data
2026-02-20 16:22:23,377 - extract - INFO - Lecture des capteurs : ../data/raw (1)/predictive_maintenance_sensor_data (1).csv
2026-02-20 16:22:23,798 - extract - INFO - Lecture des pannes : ../data/raw (1)/predictive_maintenance_failure_logs (1).csv
2026-02-20 16:22:25,360 - extract - INFO - Fichiers sauvegardés avec succès dans ../data/processed/extracted_data



✅ EXTRACTION RÉUSSIE !
Lignes capteurs chargées : 259205
Lignes pannes chargées : 23

--- Aperçu des données ---
             timestamp equipment_id equipment_type  temperature  vibration  \
0  2023-01-01 00:00:00        EQ001     compressor    65.570936   1.199164   
1  2023-01-01 00:05:00        EQ001     compressor    67.333373   1.152104   
2  2023-01-01 00:10:00        EQ001     compressor    67.720039   1.176732   

    pressure     current  
0  25.603071  120.089070  
1  25.799467  107.634332  
2  25.610601  115.977167  


In [13]:
print(sensor_df.isnull().sum())
print(f"Doublons : {sensor_df.duplicated().sum()}")

timestamp         0
equipment_id      0
equipment_type    0
temperature       0
vibration         0
pressure          0
current           0
dtype: int64
Doublons : 0


In [14]:
import pandas as pd
import numpy as np
import os
import logging
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

logger = logging.getLogger('clean')
logger.setLevel(logging.INFO)

def plot_distribution(df, column, output_path):
    plt.figure(figsize=(10, 6))
    sns.histplot(df[column], kde=True)
    plt.title(f'Distribution de {column}')
    plt.savefig(output_path)
    plt.close()

def detect_outliers(df, column, method='iqr', threshold=3):
    if method == 'zscore':
        z_scores = np.abs((df[column] - df[column].mean()) / df[column].std())
        return z_scores > threshold
    elif method == 'iqr':
        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        return (df[column] < lower_bound) | (df[column] > upper_bound)
    return pd.Series([False] * len(df))

def clean_data(input_dir='../data/processed/extracted_data', output_dir='../data/processed/cleaned_data'):
    try:
       
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        
        viz_dir = os.path.join(output_dir, 'visualizations')
        if not os.path.exists(viz_dir):
            os.makedirs(viz_dir)

       
        sensor_data_path = os.path.join(input_dir, 'sensors_extracted.csv')
        failure_data_path = os.path.join(input_dir, 'failures_extracted.csv')
        
        logger.info(f"Chargement des données depuis {input_dir}")
        sensor_df = pd.read_csv(sensor_data_path)
        failure_df = pd.read_csv(failure_data_path)
        
        original_len = len(sensor_df)

 
        sensor_df = sensor_df.replace([np.inf, -np.inf], np.nan).dropna()
        sensor_df = sensor_df.drop_duplicates()
        sensor_df['timestamp'] = pd.to_datetime(sensor_df['timestamp'])
        
      
        numeric_columns = ['temperature', 'vibration', 'pressure', 'current']
        for column in numeric_columns:
            plot_path = os.path.join(viz_dir, f'{column}_distribution.png')
            plot_distribution(sensor_df, column, plot_path)
            
            outliers_mask = detect_outliers(sensor_df, column, method='iqr')
            # On remplace par la médiane (Option 2 du code original)
            median_value = sensor_df.loc[~outliers_mask, column].median()
            sensor_df.loc[outliers_mask, column] = median_value
            logger.info(f"Outliers corrigés dans {column}: {outliers_mask.sum()}")
        
    
        failure_df['repair_cost'] = failure_df['repair_cost'].fillna(failure_df['repair_cost'].median())
        
  
        sensor_df.to_csv(os.path.join(output_dir, 'sensors_cleaned.csv'), index=False)
        failure_df.to_csv(os.path.join(output_dir, 'failures_cleaned.csv'), index=False)
        
        logger.info(f"✅ Nettoyage terminé. Données sauvées dans {output_dir}")
        return sensor_df, failure_df
    
    except Exception as e:
        logger.error(f"Erreur lors du nettoyage: {str(e)}")
        raise

if __name__ == "__main__":
 
    clean_sensor_df, clean_failure_df = clean_data()
    print("\n🚀 NETTOYAGE RÉUSSI !")
    print(clean_sensor_df.head(3))

2026-02-20 17:27:17,010 - clean - INFO - Chargement des données depuis ../data/processed/extracted_data
2026-02-20 17:27:21,431 - clean - INFO - Outliers corrigés dans temperature: 33
2026-02-20 17:27:23,229 - clean - INFO - Outliers corrigés dans vibration: 388
2026-02-20 17:27:24,389 - clean - INFO - Outliers corrigés dans pressure: 0
2026-02-20 17:27:25,978 - clean - INFO - Outliers corrigés dans current: 49
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\SkyMil\anaconda3\Lib\logging\__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\SkyMil\anaconda3\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2705' in position 41: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<


🚀 NETTOYAGE RÉUSSI !
            timestamp equipment_id equipment_type  temperature  vibration  \
0 2023-01-01 00:00:00        EQ001     compressor    65.570936   1.199164   
1 2023-01-01 00:05:00        EQ001     compressor    67.333373   1.152104   
2 2023-01-01 00:10:00        EQ001     compressor    67.720039   1.176732   

    pressure     current  
0  25.603071  120.089070  
1  25.799467  107.634332  
2  25.610601  115.977167  


In [15]:
import pandas as pd
import numpy as np
import os
import logging
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler


def create_time_features(df):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    return df

def create_rolling_features(df, window_sizes=[5, 10, 30]):
    df = df.copy().sort_values(by=['equipment_id', 'timestamp'])
    numeric_cols = ['temperature', 'vibration', 'pressure', 'current']
    for window in window_sizes:
        for col in numeric_cols:
          
            df[f'{col}_roll_mean_{window}'] = df.groupby('equipment_id')[col].transform(lambda x: x.rolling(window).mean())
            df[f'{col}_roll_std_{window}'] = df.groupby('equipment_id')[col].transform(lambda x: x.rolling(window).std())
    return df

def add_failure_indicators(sensor_df, failure_df, time_window=24):
    sensor_df = sensor_df.copy()
    sensor_df['failure_soon'] = 0
    
    failure_df['failure_timestamp'] = pd.to_datetime(failure_df['failure_timestamp'])
    
    for _, failure in failure_df.iterrows():
        equip_id = failure['equipment_id']
        fail_time = failure['failure_timestamp']
        
        # On marque à "1" les 24h avant la panne
        mask = (sensor_df['equipment_id'] == equip_id) & \
               (sensor_df['timestamp'] <= fail_time) & \
               (sensor_df['timestamp'] >= fail_time - pd.Timedelta(hours=time_window))
        sensor_df.loc[mask, 'failure_soon'] = 1
    return sensor_df

def augment_data_pipeline(input_dir='../data/processed/cleaned_data', output_dir='../data/processed/augmented_data'):
    if not os.path.exists(output_dir): os.makedirs(output_dir)
    
  
    sensor_df = pd.read_csv(os.path.join(input_dir, 'sensors_cleaned.csv'))
    failure_df = pd.read_csv(os.path.join(input_dir, 'failures_cleaned.csv'))
    
    print("🛠️ Augmentation en cours...")
    sensor_df = create_time_features(sensor_df)
    sensor_df = create_rolling_features(sensor_df)
    sensor_df = add_failure_indicators(sensor_df, failure_df)
    

    output_path = os.path.join(output_dir, 'augmented_sensor_data.csv')
    sensor_df.to_csv(output_path, index=False)
    
    return sensor_df

augmented_df = augment_data_pipeline()
print("✅ AUGMENTATION TERMINÉE !")
print(f"Nouvelles colonnes créées : {len(augmented_df.columns)}")
print(augmented_df[['timestamp', 'temperature', 'failure_soon']].tail())

🛠️ Augmentation en cours...
✅ AUGMENTATION TERMINÉE !
Nouvelles colonnes créées : 35
                 timestamp  temperature  failure_soon
259200 2023-06-29 23:40:00    64.658845             0
259201 2023-06-29 23:45:00    66.663213             0
259202 2023-06-29 23:50:00    65.587468             0
259203 2023-06-29 23:55:00    63.472214             0
259204 2023-06-30 00:00:00    68.826277             0


In [16]:
pannes_detectees = augmented_df[augmented_df['failure_soon'] == 1]
print(f"Nombre de lignes marquées comme 'Panne Proche' : {len(pannes_detectees)}")
display(pannes_detectees.head(10))

Nombre de lignes marquées comme 'Panne Proche' : 6647


,timestamp,equipment_id,equipment_type,temperature,vibration,pressure,current,hour,day_of_week,is_weekend,...,current_roll_std_10,temperature_roll_mean_30,temperature_roll_std_30,vibration_roll_mean_30,vibration_roll_std_30,pressure_roll_mean_30,pressure_roll_std_30,current_roll_mean_30,current_roll_std_30,failure_soon
24768,2023-03-28 00:00:00,EQ001,compressor,64.432818,1.176086,23.006293,120.799933,0,1,0,...,7.101277,62.404503,2.011641,1.115666,0.106215,22.315079,0.750673,103.740000,5.872248,1
24769,2023-03-28 00:05:00,EQ001,compressor,64.155820,1.202601,22.479637,107.348419,0,1,0,...,6.872968,62.643047,1.756764,1.115734,0.106273,22.263407,0.678448,103.753889,5.880545,1
24770,2023-03-28 00:10:00,EQ001,compressor,68.739770,1.318983,22.532286,122.526947,0,1,0,...,8.687689,62.870770,2.072600,1.115238,0.105248,22.273813,0.680153,104.738258,6.460669,1
24771,2023-03-28 00:15:00,EQ001,compressor,67.606775,1.282884,22.446304,120.636007,0,1,0,...,9.525569,63.135004,2.155403,1.114823,0.104537,22.244673,0.651904,105.476904,6.966990,1
24772,2023-03-28 00:20:00,EQ001,compressor,64.084253,1.071524,21.223715,106.835771,0,1,0,...,9.415378,63.229217,2.132169,1.115458,0.104202,22.180239,0.654173,105.565476,6.966806,1
24773,2023-03-28 00:25:00,EQ001,compressor,67.537116,1.149391,21.506557,113.998370,0,1,0,...,9.039009,63.493613,2.158910,1.117151,0.104332,22.133184,0.650017,105.770368,7.124951,1
24774,2023-03-28 00:30:00,EQ001,compressor,64.830342,1.348133,22.072750,110.780798,0,1,0,...,9.001186,63.647590,2.080004,1.124270,0.112525,22.125193,0.649210,106.043556,7.155665,1
24775,2023-03-28 00:35:00,EQ001,compressor,66.572540,1.303626,22.638512,105.657180,0,1,0,...,8.420686,63.835551,2.081083,1.130938,0.117091,22.174049,0.629934,106.112784,7.141044,1
24776,2023-03-28 00:40:00,EQ001,compressor,65.073932,1.053432,22.531304,103.073466,0,1,0,...,8.674211,63.903536,2.087313,1.126337,0.117342,22.166277,0.623807,105.774876,7.032615,1
24777,2023-03-28 00:45:00,EQ001,compressor,64.484301,1.179924,21.780133,107.328071,0,1,0,...,7.135675,63.980201,2.064100,1.129342,0.117528,22.166917,0.623387,105.694283,6.999314,1
